In [ ]:
import { useState, useRef } from "react";

const WELL_KNOWN = {
  20:"FTP Data", 21:"FTP Control", 22:"SSH", 23:"Telnet", 25:"SMTP",
  53:"DNS", 80:"HTTP", 110:"POP3", 143:"IMAP", 161:"SNMP", 194:"IRC",
  389:"LDAP", 443:"HTTPS", 445:"SMB", 465:"SMTPS", 587:"SMTP Submit",
  636:"LDAPS", 993:"IMAPS", 995:"POP3S", 1433:"MSSQL", 1521:"Oracle DB",
  3306:"MySQL", 3389:"RDP", 5432:"PostgreSQL", 5900:"VNC", 6379:"Redis",
  8080:"HTTP Alt", 8443:"HTTPS Alt", 8888:"Jupyter", 9200:"Elasticsearch",
  27017:"MongoDB",
};

const PRESETS = [
  { label: "Web",      ports: "80,443,8080,8443" },
  { label: "Database", ports: "3306,5432,1433,1521,27017,6379" },
  { label: "SSH/Remote", ports: "22,23,3389,5900" },
  { label: "Mail",     ports: "25,110,143,465,587,993,995" },
  { label: "Common",   ports: "21,22,25,53,80,110,143,443,3306,5432" },
];

const TIMEOUT = 1500;

async function checkPort(host, port) {
  const start = performance.now();
  try {
    const res = await fetch(`https://${host}:${port}`, {
      mode: "no-cors", signal: AbortSignal.timeout(TIMEOUT),
    });
    const latency = Math.round(performance.now() - start);
    return { port, status: "open", latency, service: WELL_KNOWN[port] || "" };
  } catch (e) {
    const msg = e?.message || "";
    const latency = Math.round(performance.now() - start);
    if (latency >= TIMEOUT - 100) {
      return { port, status: "filtered", latency, service: WELL_KNOWN[port] || "" };
    }
    if (msg.includes("Failed to fetch") || msg.includes("NetworkError")) {
      return { port, status: "open", latency, service: WELL_KNOWN[port] || "" };
    }
    return { port, status: "closed", latency, service: WELL_KNOWN[port] || "" };
  }
}

function parsePorts(raw) {
  const ports = new Set();
  for (const seg of raw.split(",")) {
    const s = seg.trim();
    if (!s) continue;
    if (s.includes("-")) {
      const [a, b] = s.split("-").map(Number);
      if (isNaN(a) || isNaN(b) || a < 1 || b > 65535 || a > b)
        throw new Error(`Invalid range: ${s}`);
      if (b - a > 199) throw new Error("Range too large (max 200 ports)");
      for (let i = a; i <= b; i++) ports.add(i);
    } else {
      const p = Number(s);
      if (isNaN(p) || p < 1 || p > 65535) throw new Error(`Invalid port: ${s}`);
      ports.add(p);
    }
  }
  if (ports.size === 0) throw new Error("No valid ports entered");
  if (ports.size > 200) throw new Error("Too many ports (max 200)");
  return [...ports].sort((a, b) => a - b);
}

const statusMeta = {
  open:     { color: "#4ade80", bg: "#052e16", label: "OPEN",     icon: "●" },
  closed:   { color: "#f87171", bg: "#2d0b0b", label: "CLOSED",   icon: "○" },
  filtered: { color: "#fbbf24", bg: "#2d1a00", label: "FILTERED", icon: "◌" },
};

export default function App() {
  const [host, setHost]       = useState("google.com");
  const [portStr, setPortStr] = useState("80,443,8080");
  const [results, setResults] = useState([]);
  const [scanning, setScanning] = useState(false);
  const [progress, setProgress] = useState(0);
  const [error, setError]     = useState("");
  const [done, setDone]       = useState(false);
  const abortRef              = useRef(false);

  async function startScan() {
    setError(""); setResults([]); setDone(false); setProgress(0);
    let ports;
    try { ports = parsePorts(portStr); }
    catch (e) { setError(e.message); return; }

    setScanning(true);
    abortRef.current = false;
    const out = [];

    for (let i = 0; i < ports.length; i++) {
      if (abortRef.current) break;
      const r = await checkPort(host.trim(), ports[i]);
      out.push(r);
      setResults([...out]);
      setProgress(Math.round(((i + 1) / ports.length) * 100));
    }

    setScanning(false);
    setDone(true);
  }

  function stopScan() { abortRef.current = true; }

  const open     = results.filter(r => r.status === "open");
  const closed   = results.filter(r => r.status === "closed");
  const filtered = results.filter(r => r.status === "filtered");

  return (
    <div style={{
      minHeight: "100vh", background: "#09090b", color: "#e4e4e7",
      fontFamily: "'JetBrains Mono', 'Fira Code', 'Cascadia Code', monospace",
      padding: "32px 16px",
    }}>
      <style>{`
        @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@300;400;600;700&display=swap');
        * { box-sizing: border-box; }
        ::selection { background: #4ade8033; }
        input { font-family: inherit; }
        button { font-family: inherit; cursor: pointer; }
        ::-webkit-scrollbar { width: 6px; }
        ::-webkit-scrollbar-track { background: #18181b; }
        ::-webkit-scrollbar-thumb { background: #3f3f46; border-radius: 3px; }
        .row-enter { animation: fadeSlide .25s ease both; }
        @keyframes fadeSlide { from { opacity:0; transform:translateX(-8px); } to { opacity:1; transform:none; } }
        .pulse { animation: pulse 1.4s ease-in-out infinite; }
        @keyframes pulse { 0%,100% { opacity:1; } 50% { opacity:.45; } }
        .scan-btn { transition: background .15s, transform .1s; }
        .scan-btn:hover:not(:disabled) { transform: translateY(-1px); }
        .scan-btn:active:not(:disabled) { transform: translateY(1px); }
        .preset-btn { transition: background .12s, color .12s; }
        .preset-btn:hover { background: #3f3f46 !important; color: #e4e4e7 !important; }
      `}</style>

      <div style={{ maxWidth: 740, margin: "0 auto" }}>

        {/* Header */}
        <div style={{ marginBottom: 32 }}>
          <div style={{ display:"flex", alignItems:"center", gap:10, marginBottom:6 }}>
            <span style={{ fontSize:22, color:"#4ade80" }}>▣</span>
            <h1 style={{ margin:0, fontSize:22, fontWeight:700, letterSpacing:"0.04em", color:"#f4f4f5" }}>
              PORT STATUS CHECKER
            </h1>
          </div>
          <p style={{ margin:0, color:"#71717a", fontSize:12, letterSpacing:"0.06em" }}>
            TCP REACHABILITY SCANNER · BROWSER-BASED
          </p>
        </div>

        {/* Input panel */}
        <div style={{
          background:"#18181b", border:"1px solid #27272a",
          borderRadius:10, padding:"24px", marginBottom:20,
        }}>
          <div style={{ display:"grid", gridTemplateColumns:"1fr 1fr", gap:16, marginBottom:16 }}>
            <label style={{ display:"flex", flexDirection:"column", gap:6 }}>
              <span style={{ fontSize:11, color:"#71717a", letterSpacing:"0.08em" }}>TARGET HOST</span>
              <input
                value={host} onChange={e => setHost(e.target.value)}
                placeholder="e.g. google.com or 192.168.1.1"
                style={{
                  background:"#09090b", border:"1px solid #3f3f46", borderRadius:6,
                  padding:"9px 12px", color:"#e4e4e7", fontSize:13, outline:"none",
                  transition:"border .15s",
                }}
                onFocus={e => e.target.style.borderColor="#4ade80"}
                onBlur={e => e.target.style.borderColor="#3f3f46"}
              />
            </label>
            <label style={{ display:"flex", flexDirection:"column", gap:6 }}>
              <span style={{ fontSize:11, color:"#71717a", letterSpacing:"0.08em" }}>PORT(S)</span>
              <input
                value={portStr} onChange={e => setPortStr(e.target.value)}
                placeholder="80, 443, 8000-8010"
                style={{
                  background:"#09090b", border:"1px solid #3f3f46", borderRadius:6,
                  padding:"9px 12px", color:"#e4e4e7", fontSize:13, outline:"none",
                  transition:"border .15s",
                }}
                onFocus={e => e.target.style.borderColor="#4ade80"}
                onBlur={e => e.target.style.borderColor="#3f3f46"}
              />
            </label>
          </div>

          {/* Presets */}
          <div style={{ display:"flex", gap:8, flexWrap:"wrap", marginBottom:20 }}>
            {PRESETS.map(p => (
              <button key={p.label} className="preset-btn"
                onClick={() => setPortStr(p.ports)}
                style={{
                  background:"#09090b", border:"1px solid #3f3f46",
                  color:"#a1a1aa", borderRadius:5, padding:"5px 11px",
                  fontSize:11, letterSpacing:"0.06em",
                }}>
                {p.label.toUpperCase()}
              </button>
            ))}
          </div>

          {error && (
            <div style={{
              background:"#2d0b0b", border:"1px solid #f8717166",
              borderRadius:6, padding:"10px 14px", color:"#f87171",
              fontSize:12, marginBottom:16,
            }}>⚠ {error}</div>
          )}

          <div style={{ display:"flex", gap:10 }}>
            <button className="scan-btn"
              onClick={startScan} disabled={scanning}
              style={{
                flex:1, background: scanning ? "#14532d" : "#16a34a",
                border:"none", color:"#fff", borderRadius:7,
                padding:"11px", fontSize:13, fontWeight:600,
                letterSpacing:"0.06em", opacity: scanning ? .7 : 1,
              }}>
              {scanning ? (
                <span className="pulse">◌ SCANNING… {progress}%</span>
              ) : "▶  START SCAN"}
            </button>
            {scanning && (
              <button className="scan-btn"
                onClick={stopScan}
                style={{
                  background:"#3f3f46", border:"none", color:"#e4e4e7",
                  borderRadius:7, padding:"11px 18px", fontSize:13,
                }}>
                ■ STOP
              </button>
            )}
          </div>
        </div>

        {/* Progress bar */}
        {(scanning || done) && (
          <div style={{
            height:3, background:"#27272a", borderRadius:2, marginBottom:20, overflow:"hidden",
          }}>
            <div style={{
              height:"100%", width:`${progress}%`, background:"#4ade80",
              transition:"width .3s ease", borderRadius:2,
            }} />
          </div>
        )}

        {/* Summary bar */}
        {results.length > 0 && (
          <div style={{
            display:"grid", gridTemplateColumns:"1fr 1fr 1fr",
            gap:10, marginBottom:20,
          }}>
            {[
              { label:"OPEN", count: open.length, color:"#4ade80", bg:"#052e16" },
              { label:"CLOSED", count: closed.length, color:"#f87171", bg:"#2d0b0b" },
              { label:"FILTERED", count: filtered.length, color:"#fbbf24", bg:"#2d1a00" },
            ].map(s => (
              <div key={s.label} style={{
                background:s.bg, border:`1px solid ${s.color}33`,
                borderRadius:8, padding:"14px 18px", textAlign:"center",
              }}>
                <div style={{ fontSize:24, fontWeight:700, color:s.color }}>{s.count}</div>
                <div style={{ fontSize:10, color:s.color+"bb", letterSpacing:"0.1em", marginTop:2 }}>{s.label}</div>
              </div>
            ))}
          </div>
        )}

        {/* Results table */}
        {results.length > 0 && (
          <div style={{
            background:"#18181b", border:"1px solid #27272a",
            borderRadius:10, overflow:"hidden",
          }}>
            <div style={{
              display:"grid", gridTemplateColumns:"80px 1fr 100px 90px",
              padding:"10px 20px", borderBottom:"1px solid #27272a",
              fontSize:10, color:"#52525b", letterSpacing:"0.1em",
            }}>
              <span>PORT</span><span>SERVICE</span><span>STATUS</span><span style={{textAlign:"right"}}>LATENCY</span>
            </div>
            <div style={{ maxHeight:420, overflowY:"auto" }}>
              {results.map((r, i) => {
                const m = statusMeta[r.status];
                return (
                  <div key={r.port} className="row-enter"
                    style={{
                      display:"grid", gridTemplateColumns:"80px 1fr 100px 90px",
                      padding:"11px 20px", alignItems:"center",
                      borderBottom:"1px solid #1f1f22",
                      background: i % 2 === 0 ? "transparent" : "#1a1a1d",
                      animationDelay: `${Math.min(i * 30, 300)}ms`,
                    }}>
                    <span style={{ color:"#e4e4e7", fontWeight:600, fontSize:13 }}>
                      {r.port}
                    </span>
                    <span style={{ color:"#71717a", fontSize:12 }}>
                      {r.service || <span style={{ color:"#3f3f46" }}>—</span>}
                    </span>
                    <span>
                      <span style={{
                        background: m.bg, color: m.color,
                        border: `1px solid ${m.color}55`,
                        borderRadius:4, padding:"2px 8px",
                        fontSize:10, fontWeight:600, letterSpacing:"0.08em",
                      }}>
                        {m.icon} {m.label}
                      </span>
                    </span>
                    <span style={{ textAlign:"right", color:"#52525b", fontSize:12 }}>
                      {r.latency != null ? `${r.latency} ms` : "—"}
                    </span>
                  </div>
                );
              })}
            </div>
          </div>
        )}

        {/* Footer note */}
        <p style={{ marginTop:24, fontSize:11, color:"#3f3f46", textAlign:"center", lineHeight:1.7 }}>
          Browser-based TCP check via no-cors fetch.<br />
          For accurate CLI scanning, use the Python script below.
        </p>
      </div>
    </div>
  );
}


In [ ]:
import socket
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
DEFAULT_TIMEOUT = 1.5   # seconds per connection attempt
MAX_WORKERS     = 50    # threads for concurrent scanning
MAX_PORT        = 65535
MIN_PORT        = 1

# Well-known service names for common ports
WELL_KNOWN = {
    20: "FTP Data",     21: "FTP Control",  22: "SSH",
    23: "Telnet",       25: "SMTP",         53: "DNS",
    67: "DHCP Server",  68: "DHCP Client",  69: "TFTP",
    80: "HTTP",         110: "POP3",        119: "NNTP",
    123: "NTP",         143: "IMAP",        161: "SNMP",
    194: "IRC",         389: "LDAP",        443: "HTTPS",
    445: "SMB",         465: "SMTPS",       514: "Syslog",
    587: "SMTP Submit", 636: "LDAPS",       993: "IMAPS",
    995: "POP3S",      1433: "MSSQL",      1521: "Oracle DB",
    3306: "MySQL",     3389: "RDP",        5432: "PostgreSQL",
    5900: "VNC",       6379: "Redis",      8080: "HTTP Alt",
    8443: "HTTPS Alt", 8888: "Jupyter",    9200: "Elasticsearch",
    27017: "MongoDB",
}

# ── Core checker ──────────────────────────────────────────────────────────
def check_port(host: str, port: int, timeout: float = DEFAULT_TIMEOUT) -> dict:
    """
    Attempt a TCP connection to host:port.

    Returns a dict with:
        host    – the target hostname/IP
        port    – the port number
        status  – 'open' | 'closed' | 'filtered'
        service – known service name or empty string
        latency – round-trip time in ms (None if not open)
        error   – error message (None if open)
    """
    result = {
        "host":    host,
        "port":    port,
        "status":  "closed",
        "service": WELL_KNOWN.get(port, ""),
        "latency": None,
        "error":   None,
    }

    start = time.monotonic()
    try:
        with socket.create_connection((host, port), timeout=timeout):
            elapsed = (time.monotonic() - start) * 1000  # ms
            result["status"]  = "open"
            result["latency"] = round(elapsed, 2)
    except socket.timeout:
        result["status"] = "filtered"
        result["error"]  = "Connection timed out"
    except ConnectionRefusedError:
        result["status"] = "closed"
        result["error"]  = "Connection refused"
    except OSError as exc:
        result["status"] = "filtered"
        result["error"]  = str(exc)

    return result


def scan_ports(host: str, ports: list[int], timeout: float = DEFAULT_TIMEOUT) -> list[dict]:
    """Scan multiple ports concurrently and return sorted results.
    """
    results = []
    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(ports))) as pool:
        futures = {pool.submit(check_port, host, p, timeout): p for p in ports}
        for future in as_completed(futures):
            results.append(future.result())
    return sorted(results, key=lambda r: r["port"])


def parse_ports(raw: str) -> list[int]:
    """
    Parse a port specification string into a list of integers.
    Accepts:
        single   – "80"
        list     – "80,443,8080"
        range    – "20-25"
        mixed    – "22,80,8000-8010"
    """
    ports = set()
    for segment in raw.split(","):
        segment = segment.strip()
        if "-" in segment:
            parts = segment.split("-", 1)
            start, end = int(parts[0]), int(parts[1])
            if not (MIN_PORT <= start <= end <= MAX_PORT):
                raise ValueError(f"Invalid port range: {segment}")
            ports.update(range(start, end + 1))
        else:
            p = int(segment)
            if not (MIN_PORT <= p <= MAX_PORT):
                raise ValueError(f"Port out of range: {p}")
            ports.add(p)
    return sorted(ports)

RESET  = "\033[0m"
GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
BOLD   = "\033[1m"
DIM    = "\033[2m"

STATUS_COLOR = {"open": GREEN, "closed": RED, "filtered": YELLOW}
STATUS_ICON  = {"open": "●", "closed": "○", "filtered": "◌"}


def print_header(host: str, ports: list[int]) -> None:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    count = len(ports)
    scope = (
        f"port {ports[0]}"
        if count == 1
        else f"{count} ports ({ports[0]}–{ports[-1]})"
    )
    print(f"\n{BOLD}{CYAN}╔══════════════════════════════════════╗{RESET}")
    print(f"{BOLD}{CYAN}║       PORT STATUS CHECKER            ║{RESET}")
    print(f"{BOLD}{CYAN}╚══════════════════════════════════════╝{RESET}")
    print(f"  {DIM}Host   :{RESET} {BOLD}{host}{RESET}")
    print(f"  {DIM}Scope  :{RESET} {scope}")
    print(f"  {DIM}Time   :{RESET} {now}\n")


def print_result(r: dict) -> None:
    color = STATUS_COLOR[r["status"]]
    icon  = STATUS_ICON[r["status"]]
    svc   = f"  {DIM}({r['service']}){RESET}" if r["service"] else ""
    lat   = f"  {DIM}{r['latency']} ms{RESET}" if r["latency"] is not None else ""
    print(f"  {color}{icon} Port {r['port']:>5}{RESET}  {color}{r['status'].upper():8}{RESET}{svc}{lat}")


def print_summary(results: list[dict]) -> None:
    open_ports     = [r for r in results if r["status"] == "open"]
    closed_ports   = [r for r in results if r["status"] == "closed"]
    filtered_ports = [r for r in results if r["status"] == "filtered"]

    print(f"\n  {DIM}{'─' * 38}{RESET}")
    print(f"  {GREEN}Open   : {len(open_ports)}{RESET}   "
          f"{RED}Closed : {len(closed_ports)}{RESET}   "
          f"{YELLOW}Filtered: {len(filtered_ports)}{RESET}")

    if open_ports:
        nums = ", ".join(str(r["port"]) for r in open_ports)
        print(f"\n  {BOLD}Open ports → {nums}{RESET}")
    print()


def interactive_mode() -> None:
    print(f"\n{BOLD}{CYAN}Port Status Checker — Interactive Mode{RESET}")
    print(f"{DIM}Press Ctrl+C to exit\n{RESET}")

    host = input("  Enter host (default: localhost): " ).strip() or "localhost"
    raw  = input("  Enter port(s) [e.g. 80, 22-25, 80,443]: " ).strip() or "80"

    try:
        ports = parse_ports(raw)
    except ValueError as e:
        print(f"\n  {RED}Error: {e}{RESET}\n")
        sys.exit(1)

    timeout_raw = input(f"  Timeout in seconds (default: {DEFAULT_TIMEOUT}): " ).strip()
    timeout = float(timeout_raw) if timeout_raw else DEFAULT_TIMEOUT

    print_header(host, ports)
    print(f"  {DIM}Scanning…{RESET}\n")

    results = scan_ports(host, ports, timeout)
    for r in results:
        print_result(r)
    print_summary(results)


def cli_mode(args: list[str]) -> None:
    if len(args) < 2:
        print(_doc_)
        sys.exit(0)

    host = args[0]
    raw  = args[1]
    timeout = float(args[2]) if len(args) >= 3 else DEFAULT_TIMEOUT

    try:
        ports = parse_ports(raw)
    except ValueError as e:
        print(f"\n  {RED}Error: {e}{RESET}\n")
        sys.exit(1)

    print_header(host, ports)
    results = scan_ports(host, ports, timeout)
    for r in results:
        print_result(r)
    print_summary(results)


if __name__ == "__main__":
    try:
        if len(sys.argv) == 1:
            interactive_mode()
        else:
            cli_mode(sys.argv[1:])
    except KeyboardInterrupt:
        print(f"\n\n  {YELLOW}Scan interrupted by user.{RESET}\n")
        sys.exit(0)
